# 噪声约束日内反转（NCIR）—赛事提交版

这是用于 BigAlpha 2026 AI 因子挖掘赛道的单因子提交 Notebook。

- 平台导入并调用 `main(datasources, start_date, end_date)`。
- 返回值严格为 `date`、`instrument`、`factor` 三列。
- 因子公式已冻结；本 Notebook 不包含回测、画图或研究辅助代码。
- 分钟数据查询上界自动扩展到结束日期的下一自然日，再裁回目标区间，避免漏掉结束日期当天的盘中数据。

在 AIStudio 中确认短区间运行无误后，直接为本 Notebook 生成 CodeShare 链接用于赛事提交。

In [1]:
def main(datasources, start_date, end_date):
    """生成 NCIR 日频因子，并且仅返回 date、instrument、factor。"""
    import numpy as np
    import pandas as pd
    import dai

    if not isinstance(datasources, dict) or "bar1m" not in datasources:
        raise ValueError("datasources 必须包含逻辑数据源 bar1m")

    start_day = pd.Timestamp(start_date).normalize()
    end_day = pd.Timestamp(end_date).normalize()
    if pd.isna(start_day) or pd.isna(end_day):
        raise ValueError("start_date 或 end_date 无法解析")
    if end_day < start_day:
        raise ValueError("end_date 不能早于 start_date")

    bar1m = datasources["bar1m"]

    # 分钟时间戳位于交易日盘中。查询到下一自然日 00:00，
    # 再把日频结果裁回 end_day，确保不遗漏结束日期当天的数据。
    minute_query_end = end_day + pd.Timedelta(days=1)
    minute_filters = {
        "date": [
            start_day.strftime("%Y-%m-%d %H:%M:%S"),
            minute_query_end.strftime("%Y-%m-%d %H:%M:%S"),
        ]
    }
    pool_filters = {
        "date": [
            start_day.strftime("%Y-%m-%d %H:%M:%S"),
            (minute_query_end - pd.Timedelta(seconds=1)).strftime(
                "%Y-%m-%d %H:%M:%S"
            ),
        ]
    }

    # 经济含义：收盘越弱但日内路径越反复，越可能是临时噪声，
    # 因而下一交易日的相对反转概率越高。
    sql = f"""
    WITH minute_base AS (
        SELECT
            date,
            instrument,
            strftime(date, '%Y-%m-%d') AS trading_day,
            close
        FROM {bar1m}
        WHERE close > 0
    ),
    minute_path AS (
        SELECT
            date,
            instrument,
            trading_day,
            close,
            lag(close, 1) OVER (
                PARTITION BY instrument, trading_day
                ORDER BY date
            ) AS previous_close
        FROM minute_base
    ),
    daily_summary AS (
        SELECT
            trading_day,
            instrument,
            avg(close) AS mean_close,
            first(close ORDER BY date) AS first_close,
            last(close ORDER BY date) AS last_close,
            sum(
                CASE
                    WHEN previous_close > 0
                    THEN abs(close / previous_close - 1)
                    ELSE 0
                END
            ) AS path_length,
            count(*) AS observation_count
        FROM minute_path
        GROUP BY trading_day, instrument
    ),
    daily_components AS (
        SELECT
            trading_day,
            instrument,
            (mean_close - last_close) / (mean_close + 1e-12)
                AS close_pressure,
            CASE
                WHEN path_length <= 1e-12 THEN 0.0
                ELSE abs(last_close / first_close - 1)
                    / (path_length + 1e-12)
            END AS raw_path_efficiency
        FROM daily_summary
        WHERE
            observation_count >= 2
            AND mean_close > 0
            AND first_close > 0
            AND last_close > 0
    )
    SELECT
        CAST(trading_day AS DATETIME) AS date,
        instrument,
        close_pressure * (
            1.0 - CASE
                WHEN raw_path_efficiency < 0 THEN 0.0
                WHEN raw_path_efficiency > 1 THEN 1.0
                ELSE raw_path_efficiency
            END
        ) AS factor
    FROM daily_components
    ORDER BY date, instrument
    """

    factor_data = dai.query(
        sql,
        filters=minute_filters,
        compression=True,
    ).df()

    required_columns = {"date", "instrument", "factor"}
    missing_columns = required_columns.difference(factor_data.columns)
    if missing_columns:
        raise ValueError(f"分钟因子查询缺少字段：{sorted(missing_columns)}")

    factor_data = factor_data[["date", "instrument", "factor"]].copy()
    factor_data["date"] = pd.to_datetime(factor_data["date"]).dt.normalize()
    factor_data["instrument"] = factor_data["instrument"].astype("string")
    factor_data["factor"] = pd.to_numeric(
        factor_data["factor"], errors="coerce"
    ).replace([np.inf, -np.inf], np.nan)
    factor_data = factor_data[
        factor_data["date"].between(start_day, end_day)
    ].copy()

    if factor_data.duplicated(["date", "instrument"]).any():
        raise ValueError("分钟聚合结果存在重复的 date + instrument")

    # 以历史中证 1000 成分表为输出骨架，缺失因子保留为 NaN，
    # 交由平台按统一规则检查覆盖率和处理。
    stock_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters=pool_filters,
        compression=True,
    ).df()
    stock_pool["date"] = pd.to_datetime(stock_pool["date"]).dt.normalize()
    stock_pool["instrument"] = stock_pool["instrument"].astype("string")
    stock_pool = stock_pool[
        stock_pool["date"].between(start_day, end_day)
    ][["date", "instrument"]].drop_duplicates()

    output = stock_pool.merge(
        factor_data,
        how="left",
        on=["date", "instrument"],
        validate="one_to_one",
    )
    return (
        output[["date", "instrument", "factor"]]
        .sort_values(["date", "instrument"])
        .reset_index(drop=True)
    )